# Question 1 - Clean Training Records Before Dataset Preparation

In [22]:
import math

def clean_training_records(records):
    if not isinstance(records, list):
        return {"cleaned": [], "rejected_indices": []}

    cleaned = []
    rejected_indices = []

    for i, record in enumerate(records):
        if not isinstance(record, dict) or 'learner_id' not in record or 'score' not in record:
            rejected_indices.append(i)
            continue

        l_id, score = record['learner_id'], record['score']

        if not isinstance(l_id, str) or not l_id.strip():
            rejected_indices.append(i)
            continue
        
        if isinstance(score, bool):
            rejected_indices.append(i)
            continue
            
        try:
            score_val = float(score)
            if not math.isfinite(score_val) or not (0 <= score_val <= 100):
                raise ValueError
        except (ValueError, TypeError):
            rejected_indices.append(i)
            continue

        cleaned.append({
            "learner_id": l_id.strip(),
            "score": score_val,
            "eligible": score_val >= 60.0
        })

    return {"cleaned": cleaned, "rejected_indices": rejected_indices}

In [23]:
records = [
    {"learner_id": " L01 ", "score": "82.5"},
    {"learner_id": "L02", "score": 60},
    {"learner_id": "L03", "score": None},
    {"learner_id": "   ", "score": 75},
    {"learner_id": "L04", "score": True},
    {"learner_id": "L05", "score": 48.0},
    {"learner_id": "L06", "score": 105},
]

clean_training_records(records)


{'cleaned': [{'learner_id': 'L01', 'score': 82.5, 'eligible': True},
  {'learner_id': 'L02', 'score': 60.0, 'eligible': True},
  {'learner_id': 'L05', 'score': 48.0, 'eligible': False}],
 'rejected_indices': [2, 3, 4, 6]}

In [24]:
#This test verifies the rejection of elements that are not dictionaries, as well as records missing the required learner_id or
#score keys. It also confirms that extra fields are ignored.

test_1 = [
    "not a dictionary", 
    {"learner_id": "L10"}, 
    {"score": 90},         
    {"learner_id": "L11", "score": 90, "extra_key": "ignoreedd"} 
]

clean_training_records(test_1)

{'cleaned': [{'learner_id': 'L11', 'score': 90.0, 'eligible': True}],
 'rejected_indices': [0, 1, 2]}

In [25]:
#This test evaluates the strict constraints on the score field, rejecting Booleans, strings 
# like "nan" and "inf" whose converted values are not finite, and values outside the 0 through 100 range.

test_2 = [
    {"learner_id": "L20", "score": False}, 
    {"learner_id": "L21", "score": "inf"}, 
    {"learner_id": "L22", "score": "nan"}, 
    {"learner_id": "L23", "score": 105},   
    {"learner_id": "L24", "score": -5.5},  
    {"learner_id": "L25", "score": 100.0}  
]

clean_training_records(test_2)


{'cleaned': [{'learner_id': 'L25', 'score': 100.0, 'eligible': True}],
 'rejected_indices': [0, 1, 2, 3, 4]}

In [26]:
#This test ensures that the learner_id is only accepted when it is a string that remains non-empty after trimming surrounding whitespace.

test_3 = [
    {"learner_id": None, "score": 50},
    {"learner_id": 123, "score": 50},
    {"learner_id": "   ", "score": 50}, 
    {"learner_id": "\t\n", "score": 50},
    {"learner_id": " L30 ", "score": 50} 
]

clean_training_records(test_3)

{'cleaned': [{'learner_id': 'L30', 'score': 50.0, 'eligible': False}],
 'rejected_indices': [0, 1, 2, 3]}

### Why does checking only whether a score is an integer risk accepting Boolean values?

### Ans . Because in Python, the Boolean values `True` and `False` are subclasses of the integer type.
### Specifically, `True` is equivalent to the integer `1`, and `False` is equivalent to the integer `0`. Therefore, if you only check whether a score is an integer, you would inadvertently accept Boolean values as valid scores, which may not be the intended behavior for your application.

# Question 2 - Prepare Mixed Text Payloads for an NLP Pipeline


In [27]:
def prepare_messages(payloads, min_length=3):
    # Enforce min_length as a positive integer strictly excluding Booleans
    if type(min_length) is not int or min_length <= 0:
        raise ValueError("min_length must be a positive integer excluding Booleans.")

    token_lists = []
    rejected_indices = []
    strip_chars = ".,!?;:"

    for i, payload in enumerate(payloads):
        # 1. Type validation and Decoding
        if isinstance(payload, str):
            text = payload
        elif isinstance(payload, (bytes, bytearray)):
            try:
                text = payload.decode('utf-8')
            except UnicodeDecodeError:
                rejected_indices.append(i)
                continue
        else:
            rejected_indices.append(i)
            continue

        # 2. Processing: lowercase, split, strip punctuation, and filter by length
        # (Uses the required list comprehension)
        tokens = [
            word.strip(strip_chars) for word in text.lower().split()
            if len(word.strip(strip_chars)) >= min_length
        ]
        token_lists.append(tokens)

    # 3. Vocabulary Construction
    # (Uses the required set comprehension to find unique tokens across all lists)
    vocabulary = sorted({token for lst in token_lists for token in lst})

    return {
        "token_lists": token_lists,
        "vocabulary": vocabulary,
        "rejected_indices": rejected_indices
    }

In [28]:
payloads_sample = [
    "  AI, helps TEAMS! ",
    b"Build reliable agents.",
    bytearray(b"AI helps."),
    None,
    b"\xff",
]
print("--- Sample Test ---")
print(prepare_messages(payloads_sample))
print()

# Test 1:"agent-based" should remain untouched inside, and length 3 words should stay.
test_1 = ["agent-based systems.", "a, b, c!"]
print("--- Test 1: Internal Punctuation ---")
print(prepare_messages(test_1, min_length=3))
print()

# Test 2: Empty inputs and whitespace Should produce empty token lists without rejecting the payload.
test_2 = ["", "   \t\n  ", b""]
print("--- Test 2: Whitespace and Empty Strings ---")
print(prepare_messages(test_2))
print()

# Test 3: Passing a Boolean or negative number must raise a ValueError.
print("--- Test 3: Invalid min_length ---")
try:
    prepare_messages(["valid string"], min_length=True)
    print("Failed: Should have raised ValueError")
except ValueError as e:
    print(f"Successfully caught ValueError: {e}")

--- Sample Test ---
{'token_lists': [['helps', 'teams'], ['build', 'reliable', 'agents'], ['helps']], 'vocabulary': ['agents', 'build', 'helps', 'reliable', 'teams'], 'rejected_indices': [3, 4]}

--- Test 1: Internal Punctuation ---
{'token_lists': [['agent-based', 'systems'], []], 'vocabulary': ['agent-based', 'systems'], 'rejected_indices': []}

--- Test 2: Whitespace and Empty Strings ---
{'token_lists': [[], [], []], 'vocabulary': [], 'rejected_indices': []}

--- Test 3: Invalid min_length ---
Successfully caught ValueError: min_length must be a positive integer excluding Booleans.


## What practical difference between bytes and bytearray matters when processing input without modifying it?  

## The practical difference between `bytes` and `bytearray`that matters when processing input without modifying it is that `bytes` is immutable, while `bytearray` is mutable. 

## When you are processing input data without the intention of modifying it, using `bytes` is generally preferred because it ensures that the data cannot be changed accidentally during processing. This immutability can help prevent bugs and maintain data integrity.